## Imports and configuration

In [45]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path(".")

FILES = {
    "short_retrieval": BASE_DIR / "retrieval_improvement_outputs" / "short_queries_117_reranked_outputs.json",
    "long_retrieval": BASE_DIR / "retrieval_improvement_outputs" / "long_queries_41_reranked_outputs.json",
    "short_eval": BASE_DIR / "failure_analysis_results" / "short_queries_117_retrieval_outputs.json",
    "long_eval": BASE_DIR / "failure_analysis_results" / "long_queries_41_retrieval_outputs.json",
}

for name, path in FILES.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

short_retrieval: FOUND
long_retrieval: FOUND
short_eval: FOUND
long_eval: FOUND


Load the four files

In [46]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

short_retrieval = load_json(FILES["short_retrieval"])
long_retrieval = load_json(FILES["long_retrieval"])
short_eval = load_json(FILES["short_eval"])
long_eval = load_json(FILES["long_eval"])

print("Short retrieval:", len(short_retrieval))
print("Long retrieval:", len(long_retrieval))
print("Short evaluation:", short_eval["total_queries"])
print("Long evaluation:", long_eval["total_queries"])

Short retrieval: 117
Long retrieval: 41


TypeError: list indices must be integers or slices, not str

Normalize evaluation data

In [38]:
def normalize_evaluation(data):
    rows = []

    for i, item in enumerate(data["results"]):
        rows.append({
            "query_id": i + 1,
            "query": item.get("query", ""),
            "expected_document": item.get("expected", ""),
            "retrieved": item.get("retrieved", []),
            "top_1": item.get("top_1", False),
            "top_3": item.get("top_3", False),
            "top_5": item.get("top_5", False),
        })

    return pd.DataFrame(rows)


short_df = normalize_evaluation(short_eval)
long_df = normalize_evaluation(long_eval)

print(short_df.shape)
print(long_df.shape)

display(short_df.head())

(117, 7)
(41, 7)


,query_id,query,expected_document,retrieved,top_1,top_3,top_5
0,1,How much vacation time do employees get?,Vacation and Sick Leave.md,"[Vacation and Sick Leave.md, Sabbatical.md, Ne...",True,True,True
1,2,Can I take leave after having a baby?,New Parent Leave.md,"[New Parent Leave.md, Other Protected Absences...",True,True,True
2,3,What holidays does the company observe?,Holiday List.md,"[Holiday List.md, README.md, Communication and...",True,True,True
3,4,How many days of PTO do I earn per month?,Vacation and Sick Leave.md,"[Vacation and Sick Leave.md, Other Protected A...",True,True,True
4,5,"I'm not feeling well today, how does sick leav...",Vacation and Sick Leave.md,"[README.md, Vacation and Sick Leave.md, Other ...",False,True,True


Remove README from evaluation results

In [39]:
def remove_readme(results):
    return [
        doc for doc in results
        if doc != "README.md"
    ]


short_df["retrieved_no_readme"] = short_df["retrieved"].apply(remove_readme)
long_df["retrieved_no_readme"] = long_df["retrieved"].apply(remove_readme)

print("README removed from evaluation rankings.")

README removed from evaluation rankings.


Calculate the improved rank

In [40]:
def get_rank(expected, retrieved):
    for rank, doc in enumerate(retrieved, start=1):
        if doc == expected:
            return rank
    return None


short_df["improved_rank"] = short_df.apply(
    lambda row: get_rank(
        row["expected_document"],
        row["retrieved_no_readme"]
    ),
    axis=1
)

long_df["improved_rank"] = long_df.apply(
    lambda row: get_rank(
        row["expected_document"],
        row["retrieved_no_readme"]
    ),
    axis=1
)

print("Short remaining Top-1 failures:",
      (short_df["improved_rank"] != 1).sum())

print("Long remaining Top-1 failures:",
      (long_df["improved_rank"] != 1).sum())

Short remaining Top-1 failures: 26
Long remaining Top-1 failures: 5


Extract only remaining failures

In [41]:
short_failures = short_df[
    short_df["improved_rank"] != 1
].copy()

long_failures = long_df[
    long_df["improved_rank"] != 1
].copy()

print("SHORT FAILURES:", len(short_failures))
print("LONG FAILURES:", len(long_failures))

SHORT FAILURES: 26
LONG FAILURES: 5


Inspect all remaining failures

In [42]:
def display_failures(df, title):
    print("=" * 100)
    print(title)
    print("=" * 100)

    for _, row in df.iterrows():

        print(f"\nQuery ID: {row['query_id']}")
        print(f"Query: {row['query']}")
        print(f"Expected: {row['expected_document']}")
        print(f"Improved Rank: {row['improved_rank']}")

        print("\nRetrieved:")
        for rank, doc in enumerate(row["retrieved_no_readme"], 1):
            marker = " <-- EXPECTED" if doc == row["expected_document"] else ""
            print(f"  {rank}. {doc}{marker}")

        print("-" * 100)


display_failures(short_failures, "SHORT QUERY REMAINING FAILURES")
display_failures(long_failures, "LONG QUERY REMAINING FAILURES")

SHORT QUERY REMAINING FAILURES

Query ID: 6
Query: Is Thanksgiving a day off at Clef?
Expected: Holiday List.md
Improved Rank: nan

Retrieved:
  1. Vacation and Sick Leave.md
  2. Other Protected Absences.md
  3. Drug and Alcohol Policy.md
  4. Welcome to Clef.md
  5. Working Remotely.md
----------------------------------------------------------------------------------------------------

Query ID: 10
Query: I need some time away, what are my options?
Expected: Vacation and Sick Leave.md
Improved Rank: nan

Retrieved:
  1. Communication and Transparency.md
  2. Working Remotely.md
  3. Other Protected Absences.md
  4. Working Remotely.md
----------------------------------------------------------------------------------------------------

Query ID: 15
Query: I've been here 5 years, what's the deal with the long break?
Expected: Sabbatical.md
Improved Rank: 2.0

Retrieved:
  1. Vacation and Sick Leave.md
  2. Sabbatical.md <-- EXPECTED
  3. Welcome to Clef.md
  4. Other Protected Absences

Create the failure-analysis framework

In [43]:
FAILURE_CATEGORIES = [
    "Ranking Failure",
    "Semantic Mismatch",
    "Vocabulary/Lexical Mismatch",
    "Cross-Document Confusion",
    "Chunking/Content Problem",
    "Low-Information Query",
    "Evaluation/Ground-Truth Issue",
    "Other"
]

print("Failure categories:")
for i, category in enumerate(FAILURE_CATEGORIES, 1):
    print(f"{i}. {category}")

Failure categories:
1. Ranking Failure
2. Semantic Mismatch
3. Vocabulary/Lexical Mismatch
4. Cross-Document Confusion
5. Chunking/Content Problem
6. Low-Information Query
7. Evaluation/Ground-Truth Issue
8. Other


Build the analysis dataframe

In [44]:
def prepare_failure_table(df, dataset):
    rows = []

    for _, row in df.iterrows():

        retrieved = row["retrieved_no_readme"]

        rows.append({
            "dataset": dataset,
            "query_id": row["query_id"],
            "query": row["query"],
            "expected_document": row["expected_document"],
            "improved_rank": row["improved_rank"],
            "rank_1_document": retrieved[0] if len(retrieved) > 0 else None,
            "rank_2_document": retrieved[1] if len(retrieved) > 1 else None,
            "rank_3_document": retrieved[2] if len(retrieved) > 2 else None,
            "rank_4_document": retrieved[3] if len(retrieved) > 3 else None,
            "rank_5_document": retrieved[4] if len(retrieved) > 4 else None,
            "failure_category": "",
            "reason": "",
            "recommended_action": "",
        })

    return pd.DataFrame(rows)


failure_analysis = pd.concat([
    prepare_failure_table(short_failures, "short"),
    prepare_failure_table(long_failures, "long")
], ignore_index=True)

print("Total remaining failures:", len(failure_analysis))

display(failure_analysis)

Total remaining failures: 31


,dataset,query_id,query,expected_document,improved_rank,rank_1_document,rank_2_document,rank_3_document,rank_4_document,rank_5_document,failure_category,reason,recommended_action
0,short,6,Is Thanksgiving a day off at Clef?,Holiday List.md,NaN,Vacation and Sick Leave.md,Other Protected Absences.md,Drug and Alcohol Policy.md,Welcome to Clef.md,Working Remotely.md,,,
1,short,10,"I need some time away, what are my options?",Vacation and Sick Leave.md,NaN,Communication and Transparency.md,Working Remotely.md,Other Protected Absences.md,Working Remotely.md,None,,,
2,short,15,"I've been here 5 years, what's the deal with t...",Sabbatical.md,2.0,Vacation and Sick Leave.md,Sabbatical.md,Welcome to Clef.md,Other Protected Absences.md,None,,,
3,short,18,Does Clef pay for online courses and books?,Continuing Education.md,2.0,Working Remotely.md,Continuing Education.md,Other Protected Absences.md,Continuing Education.md,Welcome to Clef.md,,,
4,short,36,How much do the founders make?,Salary and Equity Compensation.md,3.0,Budgeting.md,Complaint Policy.md,Salary and Equity Compensation.md,Salary and Equity Compensation.md,Objectives and Key Results.md,,,
5,short,46,"I have a family emergency, what should I do?",Other Protected Absences.md,2.0,Working Remotely.md,Other Protected Absences.md,Working Remotely.md,Communication and Transparency.md,Communication and Transparency.md,,,
6,short,70,Can we have beer at a company celebration?,Drug and Alcohol Policy.md,2.0,Welcome to Clef.md,Drug and Alcohol Policy.md,Working Remotely.md,Continuing Education.md,Policy Changes.md,,,
7,short,71,Can Clef fire me without a reason?,At-Will Employment.md,NaN,Complaint Policy.md,Employee Privacy.md,Employee Privacy.md,Other Protected Absences.md,Employee Privacy.md,,,
8,short,76,What are Clef's core values?,Clef Values.md,NaN,Welcome to Clef.md,Continuing Education.md,Equal Opportunity Employment.md,None,None,,,
9,short,78,How does Clef think about inclusion?,Clef Values.md,NaN,Equal Opportunity Employment.md,Welcome to Clef.md,Welcome to Clef.md,Welcome to Clef.md,None,,,


Add automatic failure signals

In [22]:
def classify_signal(row):

    rank = row["improved_rank"]

    if rank is not None and rank > 1:
        return "Correct document retrieved below Top-1"

    if rank is None:
        return "Expected document absent from retrieved results"

    return "Unknown"


failure_analysis["automatic_signal"] = failure_analysis.apply(
    classify_signal,
    axis=1
)

display(
    failure_analysis[
        [
            "dataset",
            "query_id",
            "query",
            "expected_document",
            "improved_rank",
            "rank_1_document",
            "automatic_signal"
        ]
    ]
)

,dataset,query_id,query,expected_document,improved_rank,rank_1_document,automatic_signal
0,short,6,Is Thanksgiving a day off at Clef?,Holiday List.md,NaN,Vacation and Sick Leave.md,Unknown
1,short,10,"I need some time away, what are my options?",Vacation and Sick Leave.md,NaN,Communication and Transparency.md,Unknown
2,short,15,"I've been here 5 years, what's the deal with t...",Sabbatical.md,2.0,Vacation and Sick Leave.md,Correct document retrieved below Top-1
3,short,18,Does Clef pay for online courses and books?,Continuing Education.md,2.0,Working Remotely.md,Correct document retrieved below Top-1
4,short,36,How much do the founders make?,Salary and Equity Compensation.md,3.0,Budgeting.md,Correct document retrieved below Top-1
5,short,46,"I have a family emergency, what should I do?",Other Protected Absences.md,2.0,Working Remotely.md,Correct document retrieved below Top-1
6,short,70,Can we have beer at a company celebration?,Drug and Alcohol Policy.md,2.0,Welcome to Clef.md,Correct document retrieved below Top-1
7,short,71,Can Clef fire me without a reason?,At-Will Employment.md,NaN,Complaint Policy.md,Unknown
8,short,76,What are Clef's core values?,Clef Values.md,NaN,Welcome to Clef.md,Unknown
9,short,78,How does Clef think about inclusion?,Clef Values.md,NaN,Equal Opportunity Employment.md,Unknown


Manually classify the failures

In [23]:
classification_options = {
    1: "Ranking Failure",
    2: "Semantic Mismatch",
    3: "Vocabulary/Lexical Mismatch",
    4: "Cross-Document Confusion",
    5: "Chunking/Content Problem",
    6: "Low-Information Query",
    7: "Evaluation/Ground-Truth Issue",
    8: "Other"
}

for i, category in classification_options.items():
    print(f"{i}. {category}")

1. Ranking Failure
2. Semantic Mismatch
3. Vocabulary/Lexical Mismatch
4. Cross-Document Confusion
5. Chunking/Content Problem
6. Low-Information Query
7. Evaluation/Ground-Truth Issue
8. Other


Save the failure analysis

In [24]:
failure_analysis.to_csv(
    "remaining_failure_analysis.csv",
    index=False,
    encoding="utf-8"
)

failure_analysis.to_json(
    "remaining_failure_analysis.json",
    orient="records",
    indent=2,
    force_ascii=False
)

print("Saved:")
print("  remaining_failure_analysis.csv")
print("  remaining_failure_analysis.json")

Saved:
  remaining_failure_analysis.csv
  remaining_failure_analysis.json


Generate category summary

In [ ]:
print("=" * 80)
print("FAILURE CATEGORY SUMMARY")
print("=" * 80)

summary = (
    failure_analysis[
        failure_analysis["failure_category"] != ""
    ]
    .groupby(["dataset", "failure_category"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset", "count"], ascending=[True, False])
)

display(summary)

Calculate percentages

In [ ]:
summary["percentage"] = (
    summary.groupby("dataset")["count"]
    .transform(lambda x: x / x.sum() * 100)
    .round(2)
)

display(summary)

Identify the dominant problem

In [ ]:
for dataset in ["short", "long"]:

    subset = summary[
        summary["dataset"] == dataset
    ]

    if len(subset) == 0:
        continue

    top = subset.iloc[0]

    print(
        f"{dataset.upper()}: "
        f"{top['failure_category']} "
        f"({top['count']} failures, "
        f"{top['percentage']}%)"
    )

Save final summary

In [ ]:
summary.to_csv(
    "failure_category_summary.csv",
    index=False,
    encoding="utf-8"
)

summary.to_json(
    "failure_category_summary.json",
    orient="records",
    indent=2,
    force_ascii=False
)

print("Final summary saved.")